# Direct YouTube Playlist MP3 Downloader

This notebook uses yt-dlp directly to download and convert YouTube playlist videos to MP3 format.

In [ ]:
!pip install yt-dlp

In [17]:
import os
import subprocess
import json
import time

In [18]:
# Create output directory if it doesn't exist
output_dir = "downloaded_mp3"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [19]:
# YouTube playlist URL
playlist_url = "https://www.youtube.com/playlist?list=PLdpwj8yUmU9mDdc4yytXlcgEAvY6sAfDA"

In [20]:
# Function to download audio from a YouTube video or playlist
def download_audio(url, output_path, format="mp3"):
    try:
        # Create the yt-dlp command with all necessary parameters
        cmd = [
            'yt-dlp',
            '--no-playlist-reverse',  # Process in default order
            '--ignore-errors',        # Skip unavailable videos
            '--no-warnings',          # Suppress warnings
            '--quiet',                # Less output
            '--progress',             # Show progress
            '-x',                     # Extract audio
            f'--audio-format={format}',  # Format
            '--audio-quality=0',      # Best quality
            # Not using cookies for now
            # Add user agent to appear more like a browser
            '--user-agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            # Add referrer
            '--referer', 'https://www.youtube.com/',
            # Add output template
            '-o', output_path,
            # Target URL
            url
        ]
        
        # Run the command and display output in real time
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, universal_newlines=True)
        
        # Print output in real time
        for line in process.stdout:
            print(line.strip())
            
        # Wait for process to complete
        process.wait()
        
        # Return success if exit code is 0
        return process.returncode == 0
    
    except Exception as e:
        print(f"Error during download: {e}")
        return False

In [21]:
# First, get information about the playlist using yt-dlp
print("Fetching playlist information...")

cmd = [
    'yt-dlp',
    '--flat-playlist',  # Don't download videos
    '--print', 'title',  # Print video titles
    playlist_url
]

try:
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    titles = result.stdout.strip().split('\n')
    video_count = len(titles)
    
    print(f"Found {video_count} videos in the playlist\n")
    for i, title in enumerate(titles[:5], 1):
        print(f"{i}. {title}")
    
    if video_count > 5:
        print(f"... and {video_count-5} more")
        
except subprocess.CalledProcessError as e:
    print(f"Error fetching playlist info: {e}\n{e.stderr}")
    video_count = "unknown"

Fetching playlist information...
Found 18 videos in the playlist

1. เธอ ๆ เพื่อนเราชอบ (Guess Who?) – SERIOUS BACON [Official MV]
2. อย่าไปรักเขาเลยเธอ (Someone Who Cares) - SERIOUS BACON [Official MV]
3. โกหกเธอทั้งนั้น (Pinocchio) - SERIOUS BACON [Official MV]
4. แฟนฉัน (Love Ads) - SERIOUS BACON [Official MV]
5. ใครทิ้งใครก่อน - SERIOUS BACON [Official MV : YOUNG PLAY]
... and 13 more


In [22]:
# Download the entire playlist
print(f"\nStarting download of {video_count} videos from playlist...")
print(f"Files will be saved to: {os.path.abspath(output_dir)}\n")

# Output template for yt-dlp
output_template = os.path.join(output_dir, "%(title)s.%(ext)s")

# Start the download
start_time = time.time()
success = download_audio(playlist_url, output_template)
end_time = time.time()

# Calculate duration
duration = end_time - start_time
minutes = int(duration // 60)
seconds = int(duration % 60)

# Show completion message
if success:
    print(f"\nDownload completed in {minutes} minutes and {seconds} seconds")
    print(f"Files are saved in: {os.path.abspath(output_dir)}")
else:
    print("\nDownload encountered some issues. Check the output above for details.")
    print("Some files may have been downloaded successfully.")


Starting download of 18 videos from playlist...
Files will be saved to: c:\Users\Chulin\FungAI\downloaded_mp3


[download]   0.0% of    3.52MiB at  999.36KiB/s ETA 00:03
[download]   0.1% of    3.52MiB at    1.94MiB/s ETA 00:01
[download]   0.2% of    3.52MiB at    4.53MiB/s ETA 00:00
[download]   0.4% of    3.52MiB at    9.72MiB/s ETA 00:00
[download]   0.9% of    3.52MiB at    5.49MiB/s ETA 00:00
[download]   1.7% of    3.52MiB at    6.47MiB/s ETA 00:00
[download]   3.5% of    3.52MiB at    8.55MiB/s ETA 00:00
[download]   7.1% of    3.52MiB at    7.66MiB/s ETA 00:00
[download]  14.2% of    3.52MiB at    8.60MiB/s ETA 00:00
[download]  28.4% of    3.52MiB at    9.59MiB/s ETA 00:00
[download]  56.7% of    3.52MiB at   10.30MiB/s ETA 00:00
[download] 100.0% of    3.52MiB at   11.41MiB/s ETA 00:00
[download] 100% of    3.52MiB in 00:00:00 at 10.62MiB/s


[download]   0.0% of    5.15MiB at  Unknown B/s ETA Unknown
[download]   0.1% of    5.15MiB at  Unknown B/s ETA Unknown
[download]   

In [25]:
# List downloaded files
try:
    files = [f for f in os.listdir(output_dir) if f.endswith('.mp3')]
    # print(f"\n{len(files)} MP3 files downloaded:")
    # for i, file in enumerate(sorted(files), 1):
    #     print(f"{i}. {file}")
except Exception as e:
    print(f"Error listing files: {e}")